# TVR

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

import os

In [2]:
import pandas as pd
from pathlib import Path

all_es = pd.read_parquet('all_es_combined.parquet')

In [3]:
# Create a new DataFrame called 'a4c_df' by filtering 'all_es'
# all_es = combined_df
a4c_df = all_es[all_es['pred_view'] == 'a4c'].copy()

# This regex finds 'echo-study', 'echo-study-1', or 'echo-study-2',
# and then captures the sequence of characters that follows until the next slash.
regex_pattern = r'results/echo-study(?:-[12])?/([^/]+)'

# Use .str.extract() to pull out the captured group (the Study ID)
a4c_df['DeidentifiedStudyID'] = a4c_df['png_uri'].str.extract(regex_pattern)

a4c_df = a4c_df[['DeidentifiedStudyID', 'mp4_uri_corrected']].rename(
    columns={'mp4_uri_corrected': 'URI'}
)

In [4]:
print(a4c_df.shape)
display(a4c_df.head())

(2976025, 2)


,DeidentifiedStudyID,URI
3,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.1714512485.1.1703111392.10353346.mp4
5,1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.3.1714512485.1.1703119859.10459774/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4
9,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111413.8215093.mp4
11,1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703111357.10353285/1.2.276.0.7230010.3.1.3.1714512485.1.1703111357.10353286/1.2.276.0.7230010.3.1.4.811753780.1.1703111387.8215053.mp4
13,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4


# Connect to Labels

In [5]:
data_labels = pd.read_csv('uhn_tvr_0912.csv')

/tmp/ipykernel_2538/3053040366.py:1: DtypeWarning: Columns (3,4,11,12,15) have mixed types. Specify dtype option on import or set low_memory=False.
  data_labels = pd.read_csv('uhn_tvr_0912.csv')


In [6]:
data_labels.head()

,Unnamed: 0,STUDY_REF,REP_ID,HLCODE,Value,Regurgitation_Binary,s3_key,STUDY_DATE,STUDY_TIME,DeidentifiedStudyID,OriginalStudyID,OriginalPatientID,DeidentifiedPatientID,SERI_ID,STUDY_ID,STUDY_INSTANCE_UID
0,0,1000846.0,NaN,NaN,none,trivial,echo-study-2/1.2.276.0.7230010.3.1.2.842097970.1.1725195860.2750974/,20150722.0,93052.0,1.2.276.0.7230010.3.1.2.842097970.1.1725195860.2750974,1.3.12.2.1107.5.8.9.1002655211149138.20150722131319960,NaN,NaN,NaN,NaN,NaN
1,1,1000714.0,NaN,NaN,trace,trivial,echo-study-2/1.2.276.0.7230010.3.1.2.895693665.1.1725195567.4281989/,20150722.0,75829.0,1.2.276.0.7230010.3.1.2.895693665.1.1725195567.4281989,1.3.12.2.1107.5.8.9.1002655211149138.20150722115621748,NaN,NaN,NaN,NaN,NaN
2,2,1000715.0,NaN,NaN,none,trivial,echo-study-2/1.2.276.0.7230010.3.1.2.895627313.1.1725195606.3247721/,20150722.0,80330.0,1.2.276.0.7230010.3.1.2.895627313.1.1725195606.3247721,1.3.12.2.1107.5.8.9.1002655211149138.20150722120229265,NaN,NaN,NaN,NaN,NaN
3,3,1000730.0,NaN,NaN,none,trivial,echo-study-2/1.2.276.0.7230010.3.1.2.1714500150.1.1725195612.2220152/,20150722.0,80354.0,1.2.276.0.7230010.3.1.2.1714500150.1.1725195612.2220152,1.3.12.2.1107.5.8.9.1002655211149138.20150722120327219,NaN,NaN,NaN,NaN,NaN
4,4,1000731.0,NaN,NaN,trace,trivial,echo-study-2/1.2.276.0.7230010.3.1.2.1667523124.1.1725195560.2343273/,20150722.0,80649.0,1.2.276.0.7230010.3.1.2.1667523124.1.1725195560.2343273,1.3.12.2.1107.5.8.9.1002655211149138.20150722114847646,NaN,NaN,NaN,NaN,NaN


In [7]:
# Select only the necessary columns from your labels DataFrame for efficiency
labels_to_merge = data_labels[['DeidentifiedStudyID', 'Regurgitation_Binary']]

# Merge the two DataFrames to find the overlap and add the 'Value'
# 'how="inner"' ensures only matching DeidentifiedStudyIDs are kept
a4c_labels = pd.merge(
    a4c_df, 
    labels_to_merge, 
    on='DeidentifiedStudyID', 
    how='inner'
)

print(f"Found {len(a4c_labels)} A4C videos.")
a4c_labels.head()

Found 2014490 A4C videos.


,DeidentifiedStudyID,URI,Regurgitation_Binary
0,1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119859.10459773/1.2.276.0.7230010.3.1.3.1714512485.1.1703119859.10459774/1.2.276.0.7230010.3.1.4.1714512485.1.1703120151.10464936.mp4,trivial
1,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.1714512485.1.1703113992.10362168.mp4,trivial
2,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.845494328.1.1703114383.13350764.mp4,trivial
3,1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703113062.10354034/1.2.276.0.7230010.3.1.3.1714512485.1.1703113062.10354035/1.2.276.0.7230010.3.1.4.859333938.1.1703114476.8024171.mp4,trivial
4,1.2.276.0.7230010.3.1.2.1714512485.1.1703119888.10460220,s3://echodata25/results/echo-study/1.2.276.0.7230010.3.1.2.1714512485.1.1703119888.10460220/1.2.276.0.7230010.3.1.3.1714512485.1.1703119888.10460221/1.2.276.0.7230010.3.1.4.845494328.1.1703120537.13449968.mp4,trivial


In [8]:
label_names = 'a4c_tvr_labels'
a4c_labels.to_csv(f"{label_names}.csv")

In [9]:
a4c_labels['Regurgitation_Binary'].value_counts()

Regurgitation_Binary
trivial                1869053
moderate_or_greater     145437
Name: count, dtype: int64

In [10]:
import pandas as pd  
from sklearn.model_selection import train_test_split  
  
# Read mitral valve regurgitation labels for A4C videos
df = pd.read_csv(f"{label_names}.csv")  
  
# Create label mapping for severity levels  
label_mapping = {  
    'trivial': 0,  
    'moderate_or_greater': 1
}  

def build_train_test(df, label_mapping):  
    # Convert string labels to integers  
    df['Value_numeric'] = df['Regurgitation_Binary'].map(label_mapping)  
      
    # Verify all labels were mapped correctly  
    if df['Value_numeric'].isna().any():  
        print("Warning: Some labels couldn't be mapped!")  
        print("Unmapped labels:", df[df['Value_numeric'].isna()]['Value'].unique())  
      
    # First split: separate out test set (60% train+val, 40% test)  
    train_val_df, test_df = train_test_split(  
        df,   
        test_size=0.2,  # 20% for test  
        stratify=df['Value_numeric'],   
        random_state=42  
    )  
      
    # Second split: split train+val into train and validation (64% train, 16% val, 20% test)  
    train_df, val_df = train_test_split(  
        train_val_df,  
        test_size=0.2,  # 20% of remaining 80% = 16% of total  
        stratify=train_val_df['Value_numeric'],  
        random_state=42  
    )  

    return train_df, val_df, test_df

In [11]:
train_df, val_df, test_df = build_train_test(df, label_mapping)

# Save files with URI and numeric labels (as expected by V-JEPA 2)  
train_df[['URI', 'Value_numeric']].to_csv(f"{label_names}_train.csv", header=False, index=False, sep=' ')  
val_df[['URI', 'Value_numeric']].to_csv(f"{label_names}_val.csv", header=False, index=False, sep=' ')  
test_df[['URI', 'Value_numeric']].to_csv(f"{label_names}_test.csv", header=False, index=False, sep=' ')  
  
# Print split statistics  
print(f"Total samples: {len(df)}")  
print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")  
print(f"Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")  
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")  
  
# Print label distribution for each split  
print("\nLabel distribution:")  
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:  
    print(f"{split_name}:")  
    for label, count in split_df['Value_numeric'].value_counts().sort_index().items():  
        original_label = [k for k, v in label_mapping.items() if v == label][0]  
        print(f"  {label} ({original_label}): {count}")  

Total samples: 2014490
Train: 1289273 (64.0%)
Validation: 322319 (16.0%)
Test: 402898 (20.0%)

Label distribution:
Train:
  0 (trivial): 1196193
  1 (moderate_or_greater): 93080
Val:
  0 (trivial): 299049
  1 (moderate_or_greater): 23270
Test:
  0 (trivial): 373811
  1 (moderate_or_greater): 29087


# Make a Full Subset

In [12]:
# --- save in V-JEPA 2 format ----------------------------------------
for name, df_small in [("train", train_df),
                       ("val",   val_df),
                       ("test",  test_df)]:
    df_small[["URI", "Value_numeric"]].to_csv(
        f"{label_names}_{name}_full.csv",
        header=False, index=False, sep=" "
    )

# Make a Tiny Subset

In [19]:
import numpy as np
import pandas as pd

def sample_per_class(df: pd.DataFrame,
                     label_col: str = "Value_numeric",
                     n_per_class: int = 500,
                     seed: int = 0) -> pd.DataFrame:
    """Balanced down-sample for fast overfit/debug runs."""
    rng = np.random.default_rng(seed)
    return (
        df.groupby(label_col, group_keys=False)
          .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=rng.integers(1e9)))
          .reset_index(drop=True)
    )

# --- choose sizes ----------------------------------------------------
train_small = sample_per_class(train_df, n_per_class=1000)   # 2 000 rows (1 000 × 2 classes)
val_small   = sample_per_class(val_df,   n_per_class=250)    #   500 rows
test_small  = sample_per_class(test_df,  n_per_class=250)    #   500 rows

# --- save in V-JEPA 2 format ----------------------------------------
for name, df_small in [("train", train_small),
                       ("val",   val_small),
                       ("test",  test_small)]:
    df_small[["URI", "Value_numeric"]].to_csv(
        f"{label_names}_{name}_tiny.csv",
        header=False, index=False, sep=" "
    )

# quick sanity check
for split, d in [("train", train_small), ("val", val_small), ("test", test_small)]:
    counts = d["Value_numeric"].value_counts().sort_index().to_dict()
    print(split, len(d), counts)

train 2000 {0: 1000, 1: 1000}
val 500 {0: 250, 1: 250}
test 500 {0: 250, 1: 250}


/tmp/ipykernel_3991/388687322.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=rng.integers(1e9)))
/tmp/ipykernel_3991/388687322.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=rng.integers(1e9)))
/tmp/ipykernel_3991/388687322.py:12: DeprecationWarning: DataFrameGroupBy.